# 03 — Train Unconditional Gan / 生成对抗网络

**Chapter 17 — File 3 of 6 / 第17章 — 第3个文件（共6个）**

---

## Summary / 总结

This script demonstrates **example of training an unconditional gan on the fashion mnist dataset**.

本脚本演示 **example of training an unconditional gan on the fashion mnist dataset**。

---
## Step 1 — example of training an unconditional gan on the fashion mnist dataset

In [ ]:
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(in_shape=(28,28,1)):
	model = Sequential()

---
## Step 3 — downsample

In [ ]:
model.add(Conv2D(128, (3,3), strides=(2,2), padding='same', input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 4 — downsample

In [ ]:
model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 5 — classifier

In [ ]:
model.add(Flatten())
	model.add(Dropout(0.4))
	model.add(Dense(1, activation='sigmoid'))

---
## Step 6 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

---
## Step 7 — define the standalone generator model

In [ ]:
def define_generator(latent_dim):
	model = Sequential()

---
## Step 8 — foundation for 7x7 image

In [ ]:
n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))

---
## Step 9 — upsample to 14x14

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 10 — upsample to 28x28

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 11 — generate

In [ ]:
model.add(Conv2D(1, (7,7), activation='tanh', padding='same'))
	return model

---
## Step 12 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(generator, discriminator):

---
## Step 13 — make weights in the discriminator not trainable

In [ ]:
discriminator.trainable = False

---
## Step 14 — connect them

In [ ]:
model = Sequential()

---
## Step 15 — add generator

In [ ]:
model.add(generator)

---
## Step 16 — add the discriminator

In [ ]:
model.add(discriminator)

---
## Step 17 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

---
## Step 18 — load fashion mnist images

In [ ]:
def load_real_samples():

---
## Step 19 — load dataset

In [ ]:
(trainX, _), (_, _) = load_data()

---
## Step 20 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(trainX, axis=-1)

---
## Step 21 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 22 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	return X

---
## Step 23 — select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 24 — choose random instances

In [ ]:
ix = randint(0, dataset.shape[0], n_samples)

---
## Step 25 — select images

In [ ]:
X = dataset[ix]

---
## Step 26 — generate class labels

In [ ]:
y = ones((n_samples, 1))
	return X, y

---
## Step 27 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples):

---
## Step 28 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 29 — reshape into a batch of inputs for the network

In [ ]:
x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

---
## Step 30 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 31 — generate points in latent space

In [ ]:
x_input = generate_latent_points(latent_dim, n_samples)

---
## Step 32 — predict outputs

In [ ]:
X = generator.predict(x_input)

---
## Step 33 — create class labels

In [ ]:
y = zeros((n_samples, 1))
	return X, y

---
## Step 34 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset.shape[0] / n_batch)
	half_batch = int(n_batch / 2)

---
## Step 35 — manually enumerate epochs

In [ ]:
for i in range(n_epochs):

---
## Step 36 — enumerate batches over the training set

In [ ]:
for j in range(bat_per_epo):

---
## Step 37 — get randomly selected 'real' samples

In [ ]:
X_real, y_real = generate_real_samples(dataset, half_batch)

---
## Step 38 — update discriminator model weights

In [ ]:
d_loss1, _ = d_model.train_on_batch(X_real, y_real)

---
## Step 39 — generate 'fake' examples

In [ ]:
X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 40 — update discriminator model weights

In [ ]:
d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)

---
## Step 41 — prepare points in latent space as input for the generator

In [ ]:
X_gan = generate_latent_points(latent_dim, n_batch)

---
## Step 42 — create inverted labels for the fake samples

In [ ]:
y_gan = ones((n_batch, 1))

---
## Step 43 — update the generator via the discriminator's error

In [ ]:
g_loss = gan_model.train_on_batch(X_gan, y_gan)

---
## Step 44 — summarize loss on this batch

In [ ]:
print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))

---
## Step 45 — save the generator model

In [ ]:
g_model.save('generator.h5')

---
## Step 46 — size of the latent space

In [ ]:
latent_dim = 100

---
## Step 47 — create the discriminator

In [ ]:
discriminator = define_discriminator()

---
## Step 48 — create the generator

In [ ]:
generator = define_generator(latent_dim)

---
## Step 49 — create the gan

In [ ]:
gan_model = define_gan(generator, discriminator)

---
## Step 50 — load image data

In [ ]:
dataset = load_real_samples()

---
## Step 51 — train model

In [ ]:
train(generator, discriminator, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of training an unconditional gan on the fashion mnist dataset 是机器学习中的常用技术。  
  *example of training an unconditional gan on the fashion mnist dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Unconditional Gan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of training an unconditional gan on the fashion mnist dataset
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Dropout

# define the standalone discriminator model
def define_discriminator(in_shape=(28,28,1)):
	model = Sequential()
	# downsample
	model.add(Conv2D(128, (3,3), strides=(2,2), padding='same', input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))
	# downsample
	model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# classifier
	model.add(Flatten())
	model.add(Dropout(0.4))
	model.add(Dense(1, activation='sigmoid'))
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

# define the standalone generator model
def define_generator(latent_dim):
	model = Sequential()
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))
	# upsample to 14x14
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 28x28
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'))
	model.add(LeakyReLU(alpha=0.2))
	# generate
	model.add(Conv2D(1, (7,7), activation='tanh', padding='same'))
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(generator, discriminator):
	# make weights in the discriminator not trainable
	discriminator.trainable = False
	# connect them
	model = Sequential()
	# add generator
	model.add(generator)
	# add the discriminator
	model.add(discriminator)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load fashion mnist images
def load_real_samples():
	# load dataset
	(trainX, _), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return X

# select real samples
def generate_real_samples(dataset, n_samples):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# select images
	X = dataset[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return X, y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	x_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	X = generator.predict(x_input)
	# create class labels
	y = zeros((n_samples, 1))
	return X, y

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=128):
	bat_per_epo = int(dataset.shape[0] / n_batch)
	half_batch = int(n_batch / 2)
	# manually enumerate epochs
	for i in range(n_epochs):
		# enumerate batches over the training set
		for j in range(bat_per_epo):
			# get randomly selected 'real' samples
			X_real, y_real = generate_real_samples(dataset, half_batch)
			# update discriminator model weights
			d_loss1, _ = d_model.train_on_batch(X_real, y_real)
			# generate 'fake' examples
			X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
			# update discriminator model weights
			d_loss2, _ = d_model.train_on_batch(X_fake, y_fake)
			# prepare points in latent space as input for the generator
			X_gan = generate_latent_points(latent_dim, n_batch)
			# create inverted labels for the fake samples
			y_gan = ones((n_batch, 1))
			# update the generator via the discriminator's error
			g_loss = gan_model.train_on_batch(X_gan, y_gan)
			# summarize loss on this batch
			print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' %
				(i+1, j+1, bat_per_epo, d_loss1, d_loss2, g_loss))
	# save the generator model
	g_model.save('generator.h5')

# size of the latent space
latent_dim = 100
# create the discriminator
discriminator = define_discriminator()
# create the generator
generator = define_generator(latent_dim)
# create the gan
gan_model = define_gan(generator, discriminator)
# load image data
dataset = load_real_samples()
# train model
train(generator, discriminator, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 4 of 6